In [28]:
# Data Cleaning
# Keeping only the relevant columns and checking for invalid or missing data.

import pandas as pd

income = pd.read_csv("../dataset/raw/income.csv")
balance = pd.read_csv("../dataset/raw/balance.csv") 
cashflow = pd.read_csv("../dataset/raw/cashflow.csv")
prices = pd.read_csv("../dataset/raw/prices.csv")
companies = pd.read_csv("../dataset/raw/companies.csv")

In [ ]:
# Checking the columns of the income statement
print(income.columns.tolist())

# Selecting relevant columns from the income statement
income_cols = ["Ticker", "Fiscal Year", "Report Date", "Revenue",
               "Operating Income (Loss)", "Interest Expense, Net",
               "Net Income", "Shares (Basic)", "Cost of Revenue"]
income = income[income_cols]

# Checking for missing values, gaps, and invalid data in the income statement
gaps = income.sort_values(["Ticker", "Fiscal Year"]).groupby("Ticker")["Fiscal Year"].diff()
problem = (
    (gaps > 1)
    | income.isna().any(axis=1)
    | (income["Revenue"] <= 0)
    | (income["Shares (Basic)"] <= 0)
)
print(income[problem])

# Fixing missing values in the income statement
income = income.fillna(0)

print(income['Ticker'].nunique())

# After filling missing values:
print(income[income.isna().any(axis=1)])

['Ticker', 'Report Date', 'SimFinId', 'Currency', 'Fiscal Year', 'Fiscal Period', 'Publish Date', 'Restated Date', 'Shares (Basic)', 'Shares (Diluted)', 'Revenue', 'Cost of Revenue', 'Gross Profit', 'Operating Expenses', 'Selling, General & Administrative', 'Research & Development', 'Depreciation & Amortization', 'Operating Income (Loss)', 'Non-Operating Income (Loss)', 'Interest Expense, Net', 'Pretax Income (Loss), Adj.', 'Abnormal Gains (Losses)', 'Pretax Income (Loss)', 'Income Tax (Expense) Benefit, Net', 'Income (Loss) from Continuing Operations', 'Net Extraordinary Gains (Losses)', 'Net Income', 'Net Income (Common)']
   Ticker  Fiscal Year Report Date       Revenue  Operating Income (Loss)  \
40   INTC         2020  2020-12-31  7.786700e+10             2.387600e+10   
41   INTC         2021  2021-12-31  7.902400e+10             2.208200e+10   
42   INTC         2022  2022-12-31  6.305400e+10             2.336000e+09   
43   INTC         2023  2023-12-31  5.422800e+10           

In [30]:
# Checking the columns of the balance sheet
print(balance.columns.tolist())

# Selecting relevant columns from the balance sheet
balance_cols = ["Ticker", "Fiscal Year", "Report Date",
                "Cash, Cash Equivalents & Short Term Investments", "Inventories",
                "Total Current Assets", "Total Assets",
                "Short Term Debt", "Total Current Liabilities", "Long Term Debt",
                "Total Liabilities", "Retained Earnings", "Total Equity",
                "Total Liabilities & Equity", "Accounts & Notes Receivable"]
balance = balance[balance_cols]

# Checking for missing values, gaps, and invalid data in the balance sheet
balance = balance.sort_values(["Ticker", "Fiscal Year"])
gaps = balance.groupby("Ticker")["Fiscal Year"].diff()

core = ["Total Assets", "Total Current Assets", "Total Current Liabilities",
        "Total Liabilities", "Total Equity", "Retained Earnings",
        "Cash, Cash Equivalents & Short Term Investments",
        'Accounts & Notes Receivable']

identity_diff = (balance["Total Assets"] - balance["Total Liabilities & Equity"]).abs()

problem = (
    (gaps > 1)
    | balance[core].isna().any(axis=1)
    | (balance["Total Current Liabilities"] <= 0)
    | (identity_diff > 0.01)
    | balance.duplicated(["Ticker", "Fiscal Year"], keep=False)
)
print(balance[problem])

# Negative equity does not break the data, but it distorts ROE and debt/equity
print(balance[balance["Total Equity"] <= 0][["Ticker", "Fiscal Year", "Total Equity"]])
# There are companies with negative equity, so we will keep attention to them when analyzing the data.

# Fixing missing values: 0 only where NaN means "none"
zero_cols = ["Short Term Debt", "Long Term Debt", "Inventories"]
balance[zero_cols] = balance[zero_cols].fillna(0)
balance = balance.dropna(subset=core)

print(balance['Ticker'].nunique())

# After fixing:
print(balance[balance.isna().any(axis=1)])

['Ticker', 'Report Date', 'SimFinId', 'Currency', 'Fiscal Year', 'Fiscal Period', 'Publish Date', 'Restated Date', 'Shares (Basic)', 'Shares (Diluted)', 'Cash, Cash Equivalents & Short Term Investments', 'Accounts & Notes Receivable', 'Inventories', 'Total Current Assets', 'Property, Plant & Equipment, Net', 'Long Term Investments & Receivables', 'Other Long Term Assets', 'Total Noncurrent Assets', 'Total Assets', 'Payables & Accruals', 'Short Term Debt', 'Total Current Liabilities', 'Long Term Debt', 'Total Noncurrent Liabilities', 'Total Liabilities', 'Share Capital & Additional Paid-In Capital', 'Treasury Stock', 'Retained Earnings', 'Total Equity', 'Total Liabilities & Equity']
   Ticker  Fiscal Year Report Date  \
2    AMZN         2022  2022-12-31   
25     DG         2020  2021-01-31   
26     DG         2021  2022-01-31   
27     DG         2022  2023-01-31   
28     DG         2023  2024-01-31   
29     DG         2024  2025-01-31   
30   DLTR         2020  2021-01-31   
31   

In [ ]:
# Checking the columns of the cashflow statement
print(cashflow.columns.tolist())

cashflow_cols = ["Ticker", "Fiscal Year", "Report Date",
                 "Net Cash from Operating Activities",
                 "Change in Fixed Assets & Intangibles",
                 "Depreciation & Amortization",
                 "Dividends Paid",
                 "Cash from (Repayment of) Debt",
                 "Cash from (Repurchase of) Equity"]
cashflow = cashflow[cashflow_cols]

cashflow["CapEx"] = cashflow["Change in Fixed Assets & Intangibles"].abs()
cashflow["Free Cash Flow"] = cashflow["Net Cash from Operating Activities"] - cashflow["CapEx"]

core = ["Net Cash from Operating Activities", "Depreciation & Amortization"]

# Checking for missing values, gaps, and invalid data in the cashflow statement
cashflow = cashflow.sort_values(["Ticker", "Fiscal Year"]).reset_index(drop=True)
gaps = cashflow.groupby("Ticker")["Fiscal Year"].diff()

problem = (
    (gaps > 1)
    | cashflow[core].isna().any(axis=1)
    | cashflow.duplicated(["Ticker", "Fiscal Year"], keep=False)
    | (cashflow["Depreciation & Amortization"] < 0)
)

print(cashflow['Ticker'].nunique())
# After fixing:
print(cashflow[problem])

['Ticker', 'Report Date', 'SimFinId', 'Currency', 'Fiscal Year', 'Fiscal Period', 'Publish Date', 'Restated Date', 'Shares (Basic)', 'Shares (Diluted)', 'Net Income/Starting Line', 'Depreciation & Amortization', 'Non-Cash Items', 'Change in Working Capital', 'Change in Accounts Receivable', 'Change in Inventories', 'Change in Accounts Payable', 'Change in Other', 'Net Cash from Operating Activities', 'Change in Fixed Assets & Intangibles', 'Net Change in Long Term Investment', 'Net Cash from Acquisitions & Divestitures', 'Net Cash from Investing Activities', 'Dividends Paid', 'Cash from (Repayment of) Debt', 'Cash from (Repurchase of) Equity', 'Net Cash from Financing Activities', 'Net Change in Cash']
Empty DataFrame
Columns: [Ticker, Fiscal Year, Report Date, Net Cash from Operating Activities, Change in Fixed Assets & Intangibles, Depreciation & Amortization, Dividends Paid, Cash from (Repayment of) Debt, Cash from (Repurchase of) Equity, CapEx, Free Cash Flow]
Index: []


In [ ]:
# Checking the columns of the prices dataset
print(prices.columns.tolist())

prices_cols = ["Ticker", "Date", "Close", "Adj. Close", "Shares Outstanding", "Dividend"]

prices = prices[prices_cols]
prices["Date"] = pd.to_datetime(prices["Date"])

# Checking for missing values, gaps, and invalid data in the prices dataset
problem = (
    prices[["Close", "Adj. Close", "Shares Outstanding"]].isna().any(axis=1)
    | (prices["Close"] <= 0)
    | (prices["Shares Outstanding"] <= 0)
    | (prices['Adj. Close'] <= 0)
    | (prices.duplicated(["Ticker", "Date"], keep=False))
)

print(prices['Ticker'].nunique())
# After fixing:
print(prices[problem])

['Ticker', 'Date', 'SimFinId', 'Open', 'High', 'Low', 'Close', 'Adj. Close', 'Volume', 'Dividend', 'Shares Outstanding']
Empty DataFrame
Columns: [Ticker, Date, Close, Adj. Close, Shares Outstanding, Dividend]
Index: []


In [ ]:
# Checking the columns of the companies dataset
print(companies.columns.tolist())

companies_cols = ["Ticker", "Company Name", "Sector", "Industry", "End of financial year (month)"]
companies = companies[companies_cols]

# Checking for missing values, gaps, and invalid data in the companies dataset
problem = (
    companies.isna().any(axis=1)
    | (companies.duplicated(["Ticker"], keep=False))
    | (companies["End of financial year (month)"] < 1)
    | (companies["End of financial year (month)"] > 12)
)
companies['End of financial year (month)'] = companies['End of financial year (month)'].astype('Int64')  # Convert to nullable integer type

print(companies['Ticker'].nunique())
print(companies[problem])

['Ticker', 'SimFinId', 'Company Name', 'IndustryId', 'ISIN', 'End of financial year (month)', 'Number Employees', 'Business Summary', 'Market', 'CIK', 'Main Currency', 'Industry', 'Sector']
Empty DataFrame
Columns: [Ticker, Company Name, Sector, Industry, End of financial year (month)]
Index: []


In [34]:
# Saving the cleaned datasets
income.to_csv("../dataset/processed/income_cleaned.csv", index=False)
balance.to_csv("../dataset/processed/balance_cleaned.csv", index=False)
cashflow.to_csv("../dataset/processed/cashflow_cleaned.csv", index=False)
prices.to_csv("../dataset/processed/prices_cleaned.csv", index=False)
companies.to_csv("../dataset/processed/companies_cleaned.csv", index=False)